# Entrenamiento de CNN con Transfer Learning
## Objetivos
- Baseline simple con transfer learning usando ResNet18 o VGG16,
- Entrenar solo la cabeza clasificadora (fine-tuning parcial),
- Hacer fine-tuning parcial o total según resultados,
- Guardar métricas por época,
- Guardar el mejor modelo según validación,
- Matriz de confusión y reporte de clasificación final,
- Accuracy, precision, recall y F1-score.

In [1]:
# Importaciones
import os

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from torchvision import models
from torchvision import datasets
from torchvision import transforms as tfs

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport

In [2]:
# Configuración general
PATHS = setup()
report = ProcessingReport()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 16
EPOCHS = 75
LR = 1e-4
UNFREEZE_EPOCH = 10
BACKBONE_LR_FACTOR = 0.1
BACKBONE_LR = LR * BACKBONE_LR_FACTOR

NUM_WORKERS = min(8, os.cpu_count() or 1)
PIN_MEMORY = DEVICE.type == "cuda"
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16
USE_CHANNELS_LAST = DEVICE.type == "cuda"
ENABLE_MODEL_COMPILE = False

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

IMG_SIZE = 224

FLIP_PROBABILITY = 0.5
SCALING_FACTOR_INTERVAL = (0.8, 1.)
ROTATION_DEGREES = (-15, 15)
PIXEL_FILL = (255, 255, 255)

SEED = 42

DATA_DIR = PATHS.DATA_UNIFICADO
MODEL_NAME = "resnet18"

print("Device:", DEVICE)
print(
    f"Config -> batch_size: {BATCH_SIZE}, workers: {NUM_WORKERS}, "
    f"pin_memory: {PIN_MEMORY}, amp: {USE_AMP}"
)

Device: cuda
Config -> batch_size: 16, workers: 8, pin_memory: True, amp: True


In [3]:
# Transformaciones
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = tfs.Compose([
    tfs.Resize((IMG_SIZE, IMG_SIZE), interpolation=tfs.InterpolationMode.BILINEAR),
    tfs.RandomHorizontalFlip(p=FLIP_PROBABILITY),
    tfs.RandomRotation(degrees=ROTATION_DEGREES, fill=PIXEL_FILL),
    tfs.RandomAffine(degrees=0, scale=SCALING_FACTOR_INTERVAL, fill=PIXEL_FILL),
    tfs.ToTensor(),
    tfs.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = tfs.Compose([
    tfs.Resize((IMG_SIZE, IMG_SIZE), interpolation=tfs.InterpolationMode.BILINEAR),
    tfs.ToTensor(),
    tfs.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_transforms = val_transforms


class TransformSubset(Dataset):
    """Subset por indices que aplica transform al vuelo."""

    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        x, y = self.dataset[self.indices[i]]  # x es PIL si dataset base no tiene transform
        if self.transform is not None:
            x = self.transform(x)
        return x, y


# a) Dataset base SIN transformaciones
full_dataset = datasets.ImageFolder(DATA_DIR)

indices = np.arange(len(full_dataset))
labels = np.array(full_dataset.targets)

# train vs temp
train_idx, temp_idx, train_labels, temp_labels = train_test_split(
    indices,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=SEED
)

# val vs test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_labels,
    random_state=SEED
)

train_dataset = TransformSubset(full_dataset, train_idx, transform=train_transforms)
val_dataset = TransformSubset(full_dataset, val_idx, transform=val_transforms)
test_dataset = TransformSubset(full_dataset, test_idx, transform=test_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)

class_names = full_dataset.classes
NUM_CLASSES = len(class_names)

# Pesos inversos a la frecuencia para reducir el sesgo hacia clases mayoritarias.
train_class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = train_class_counts.sum() / (NUM_CLASSES * np.maximum(train_class_counts, 1))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

print("Clases:", class_names)
print("Conteo train por clase:", train_class_counts.tolist())
print("Pesos de clase:", class_weights.round(3).tolist())
print(f"Split -> train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")

Clases: ['clase01', 'clase02', 'clase03', 'clase04', 'clase05', 'clase06', 'clase07', 'clase08', 'clase09', 'clase10', 'clase11']
Conteo train por clase: [364, 644, 994, 140, 784, 714, 672, 3920, 280, 4970, 1652]
Pesos de clase: [3.78, 2.136, 1.384, 9.827, 1.755, 1.927, 2.047, 0.351, 4.914, 0.277, 0.833]
Split -> train: 15134, val: 3243, test: 3243


In [4]:
from torchvision.models import ResNet18_Weights


# 3. Visualización rápida (sanity check)
def show_batch(loader):
    images, labels = next(iter(loader))
    images = images[:8]
    labels = labels[:8]

    fig, axes = plt.subplots(1, 8, figsize=(16, 3))
    for i, ax in enumerate(axes):
        img = images[i].permute(1, 2, 0).numpy()
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(class_names[labels[i]])
        ax.axis("off")


# Modelo (reemplaza rama resnet18 en get_model)
def get_model(model_name, num_classes):
    if model_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT  # equivalente a IMAGENET1K_V1
        model = models.resnet18(weights=weights, progress=True)

        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "vgg16":
        model = models.vgg16(weights="IMAGENET1K_V1")
        for param in model.parameters():
            param.requires_grad = False
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

    else:
        raise ValueError("Modelo no soportado")

    return model.to(DEVICE)


model = get_model(MODEL_NAME, NUM_CLASSES)
if USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)

if ENABLE_MODEL_COMPILE and hasattr(torch, "compile") and DEVICE.type == "cuda":
    model = torch.compile(model)

In [5]:
# 5. Funciones de 03.entrenamiento y validación
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
scaler = torch.amp.GradScaler(device=DEVICE.type, enabled=USE_AMP)

# Optimizador (ajusta para entrenar solo lo descongelado)
optimizer = optim.Adam(
    [{"params": [p for p in model.parameters() if p.requires_grad], "lr": LR, "name": "head"}],
    lr=LR,
)


def maybe_unfreeze_layer4(model, optimizer, epoch):
    if MODEL_NAME != "resnet18" or epoch != UNFREEZE_EPOCH or not hasattr(model, "layer4"):
        return False

    if any(group.get("name") == "layer4" for group in optimizer.param_groups):
        return False

    layer4_params = []
    for param in model.layer4.parameters():
        param.requires_grad = True
        layer4_params.append(param)

    optimizer.add_param_group({"params": layer4_params, "lr": BACKBONE_LR, "name": "layer4"})
    return True


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=PIN_MEMORY)
        labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
        if USE_CHANNELS_LAST:
            images = images.to(memory_format=torch.channels_last)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc


def validate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=PIN_MEMORY)
            labels = labels.to(DEVICE, non_blocking=PIN_MEMORY)
            if USE_CHANNELS_LAST:
                images = images.to(memory_format=torch.channels_last)

            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = correct / len(loader.dataset)
    return total_loss / len(loader), acc, all_preds, all_labels

In [ ]:
# 6. Loop de 03.entrenamiento
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(EPOCHS):
    if maybe_unfreeze_layer4(model, optimizer, epoch):
        print(
            f"Epoch {epoch + 1}: se descongela layer4 con lr={BACKBONE_LR:.2e} "
            f"(head lr={LR:.2e})"
        )

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc, preds, labels = validate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/75
Train Loss: 2.2420 | Train Acc: 0.3246
Val Loss: 2.2259 | Val Acc: 0.4012
Epoch 2/75
Train Loss: 2.0263 | Train Acc: 0.4267
Val Loss: 2.1412 | Val Acc: 0.3907
Epoch 3/75
Train Loss: 1.9175 | Train Acc: 0.4577
Val Loss: 2.0562 | Val Acc: 0.3996
Epoch 4/75
Train Loss: 1.8492 | Train Acc: 0.4739
Val Loss: 2.0314 | Val Acc: 0.4576
Epoch 5/75
Train Loss: 1.7962 | Train Acc: 0.4852
Val Loss: 1.9653 | Val Acc: 0.4406
Epoch 6/75
Train Loss: 1.7562 | Train Acc: 0.4930
Val Loss: 1.9590 | Val Acc: 0.3969
Epoch 7/75
Train Loss: 1.7326 | Train Acc: 0.4961
Val Loss: 1.9552 | Val Acc: 0.3981
Epoch 8/75
Train Loss: 1.7042 | Train Acc: 0.4982
Val Loss: 1.8776 | Val Acc: 0.4055
Epoch 9/75
Train Loss: 1.6758 | Train Acc: 0.5017
Val Loss: 1.8973 | Val Acc: 0.4086
Epoch 10/75
Train Loss: 1.6665 | Train Acc: 0.4976
Val Loss: 1.8436 | Val Acc: 0.4493
Epoch 11: se descongela layer4 con lr=1.00e-05 (head lr=1.00e-04)
Epoch 11/75
Train Loss: 1.4658 | Train Acc: 0.5416
Val Loss: 1.4144 | Val Acc: 0.56

In [ ]:
# 7. Curvas de 03.entrenamiento
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss")
plt.show()

plt.plot(train_accs, label="Train Acc")
plt.plot(val_accs, label="Val Acc")
plt.legend()
plt.title("Accuracy")
plt.show()

In [ ]:
# 8. Evaluación final
_, test_acc, final_preds, final_labels = validate(model, test_loader)
print(f"Test Acc: {test_acc:.4f}")

all_label_ids = list(range(NUM_CLASSES))

print(
    classification_report(
        final_labels,
        final_preds,
        labels=all_label_ids,
        target_names=class_names,
        zero_division=0,
    )
)

cm = confusion_matrix(final_labels, final_preds, labels=all_label_ids, normalize='true')
sns.heatmap(cm, annot=True, fmt=".2f", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# 9. Guardado del modelo
torch.save(model.state_dict(), PATHS.OUTPUT_MODELS / f"{MODEL_NAME}_final.pth")